In [1]:
from pathlib import Path
import sys
import os

# 1. Get the absolute path of the directory containing this script
parent_dir = Path(os.getcwd()).resolve().parent

# 3. Add the parent directory to Python's search path
sys.path.append(str(parent_dir))

##############################################################################
# 4. Import the function normally
from ingest import load_faq_data
import rag_helper
import evaluation_utils

# Use the function
documents = load_faq_data()

In [2]:
# We want to look only into FAQ pertaining to LLM course

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

documents = documents_llm

doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [5]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [6]:
import json

user_prompt = json.dumps(doc)
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [7]:
# Creating the mssage

messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

# Instead of calling 'responses.create' and read 'response,output_text',
# we use 'responses.parse' and pass 'text_format=Questions' to get a structured output
# the API will then return our class instead of free text
response = openai_client.responses.parse(
    model = 'openai/gpt-oss-20b',
    input = messages,
    text_format=Questions
)

result = response.output_parsed

print(result)

questions=['Can I enroll now after seeing the course posted?', 'If I enroll late, will I get a certificate?', 'What’s the deadline for submitting the final project?', 'Is there a cutoff date for project submission to earn the certificate?', 'Do I need to submit the project before the course ends to qualify for the certificate?']


In [8]:
print(result.questions)

['Can I enroll now after seeing the course posted?', 'If I enroll late, will I get a certificate?', 'What’s the deadline for submitting the final project?', 'Is there a cutoff date for project submission to earn the certificate?', 'Do I need to submit the project before the course ends to qualify for the certificate?']


In [9]:
type(result.questions)

list

In [10]:
# This code was already encapsulated in a function in evaluation_utils.py, so we can use it directly
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['When can I join the LLM Zoomcamp after discovering it?', 'Will I be able to get a certificate if I enroll late?', 'What do I need to do to receive a certificate after joining the course?', 'Is there a deadline for submitting my project to earn the certificate?', 'Can I still finish the course and get certified if I start now?']


In [11]:
# from response.usage
usage.input_tokens, usage.output_tokens

(336, 532)

In [12]:
from evaluation_utils import calc_price
cost = calc_price(usage)

cost

{'input_cost': 0.000252,
 'output_cost': 0.0023940000000000003,
 'total_cost': 0.0026460000000000003}

In [13]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'When can I join the LLM Zoomcamp after discovering it?',
  'document': '74eb249bbf'},
 {'question': 'Will I be able to get a certificate if I enroll late?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to receive a certificate after joining the course?',
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline for submitting my project to earn the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I still finish the course and get certified if I start now?',
  'document': '74eb249bbf'}]

# Scale up the synthetic data generation
So far, we have fenerated questions for one document, now we do that for every document in the FAQ dataset.

In [14]:
from evaluation_utils import llm_structured_retry

# llm_structured_retry is a wrapper around llm_structured that retries the call if it fails
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [15]:
# Try this for the first 5 documents
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

# Parallel Processing

In [16]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

# as generate_ground_truth() returns 2 things: generated records and the token usage
# results is an iteratable of (records, usage)

ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01ktn3152kfxksvjzdq0a1djkp` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7398, Requested 995. Please try again in 2.9475s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [18]:
# Since I am using the free tier of Groq and hit a rate limit for the model,
# I would want to run this document by document with a delay between each call.
# Try this for the first 5 documents
import time
    
ground_truth = []
usages = []

for doc in tqdm(documents):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)
    time.sleep(5)  # Add a 5-second delay between each call

  0%|          | 0/113 [00:00<?, ?it/s]

In [19]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

# the same as 
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.29946075000000005

In [21]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth

,question,document
0,Can I join the course now that it's already ru...,74eb249bbf
1,How do I enroll if I discovered the course jus...,74eb249bbf
2,Will I be able to get a certificate if I sign ...,74eb249bbf
3,Do I need to finish and submit my project to r...,74eb249bbf
4,What’s the deadline for project submission if ...,74eb249bbf
...,...,...
560,"My requests library keeps installing v2.28, bu...",4b30b918bc
561,Why is pip installing an older requests versio...,4b30b918bc
562,I’m getting a 401 error when using lancedb; co...,4b30b918bc
563,How can I manually install requests v2.32.3 fo...,4b30b918bc


In [23]:
# Save it for later úe
df_ground_truth.to_csv("ground_truth-new.csv", index=False)